# 07 — Ablation: Tree-Recurrent vs. Chain-Recurrent

**Reddit Opinion & Community Research — direct test of the paper's central architectural claim**


In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("torch", None), ("numpy", None), ("scikit-learn", "sklearn"), ("scipy", None)]:
    ensure(pkg, imp)

print("Packages OK.")


Packages OK.


In [2]:
import os, json, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

with open(os.path.join(BASE_DIR, "processed_v2", "dataset", "dataset_contract.json"), "r", encoding="utf-8") as f:
    dataset_contract = json.load(f)
DATASET_DIR = dataset_contract["dataset_dir"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: switch to a GPU runtime -- 10 training runs will be very slow on CPU.")


Mounted at /content/drive
Using device: cuda


## Load dataset 

In [3]:
def load_split(name):
    with open(os.path.join(DATASET_DIR, f"{name}_items.pkl"), "rb") as f:
        return pickle.load(f)

splits = {name: load_split(name) for name in ["train", "val", "test"]}
embeddings = np.load(dataset_contract["embeddings_path"]).astype("float32")
node_feat_matrix = np.load(os.path.join(DATASET_DIR, "node_feat_matrix.npy"))

with open(dataset_contract["vocab_path"], "r", encoding="utf-8") as f:
    vocab = json.load(f)
N_SENTIMENT, N_EMOTION = len(vocab["sentiment_to_idx"]), len(vocab["emotion_to_idx"])

for name, split in splits.items():
    print(f"{name}: {len(split):,} hyperedges")


train: 6,440 hyperedges
val: 1,380 hyperedges
test: 1,380 hyperedges


## Collate + batching 

In [4]:
def make_collate_fn(embeddings, node_feat_matrix):
    emb_dim = embeddings.shape[1]
    n_node_feats = node_feat_matrix.shape[1]

    def collate(batch):
        B = len(batch)
        L = max(it["length"] for it in batch)
        emb_batch = np.zeros((B, L, emb_dim), dtype="float32")
        node_feat_batch = np.zeros((B, L, n_node_feats), dtype="float32")
        parent_batch = np.full((B, L), -1, dtype="int64")
        delta_t_batch = np.zeros((B, L), dtype="float32")
        sentiment_batch = np.full((B, L), -1, dtype="int64")
        emotion_batch = np.full((B, L), -1, dtype="int64")
        mask = np.zeros((B, L), dtype="bool")

        for i, it in enumerate(batch):
            n = it["length"]
            emb_batch[i, :n] = embeddings[it["emb_idx"]]
            valid_author = it["author_idx"] >= 0
            node_feat_batch[i, :n][valid_author] = node_feat_matrix[it["author_idx"][valid_author]]
            parent_batch[i, :n] = it["parent_idx"]
            delta_t_batch[i, :n] = it["delta_t"]
            sentiment_batch[i, :n] = it["sentiment_idx"]
            emotion_batch[i, :n] = it["emotion_idx"]
            mask[i, :n] = True

        return {
            "embeddings": torch.from_numpy(emb_batch), "node_features": torch.from_numpy(node_feat_batch),
            "parent_idx": torch.from_numpy(parent_batch), "delta_t": torch.from_numpy(delta_t_batch),
            "sentiment_idx": torch.from_numpy(sentiment_batch), "emotion_idx": torch.from_numpy(emotion_batch),
            "mask": torch.from_numpy(mask),
        }
    return collate

collate_fn = make_collate_fn(embeddings, node_feat_matrix)


def make_length_bucketed_batches(items, batch_size, rng):
    lengths = np.array([it["length"] for it in items])
    order = np.argsort(lengths)
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    rng.shuffle(batches)
    return batches


def make_chain_idx(mask):
    B, L = mask.shape
    idx = torch.arange(L, device=mask.device).unsqueeze(0).expand(B, L).clone() - 1
    return torch.where(idx < 0, torch.full_like(idx, -1), idx)


## Model 

In [5]:
class TreeRecurrentSSMLayer(nn.Module):
    def __init__(self, d_model, state_dim=16):
        super().__init__()
        self.d_model, self.state_dim = d_model, state_dim
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.1)
        self.W_delta = nn.Linear(d_model, d_model)
        self.W_B = nn.Linear(d_model, state_dim)
        self.W_C = nn.Linear(d_model, state_dim)
        self.D = nn.Parameter(torch.randn(d_model) * 0.1)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, parent_idx, mask):
        B, L, D = x.shape
        N = self.state_dim
        A = -torch.exp(self.A_log)
        delta = F.softplus(self.W_delta(x))
        Bt = self.W_B(x)
        Ct = self.W_C(x)
        Abar = torch.exp(delta.unsqueeze(-1) * A.view(1, 1, D, N))
        Bbar = delta.unsqueeze(-1) * Bt.unsqueeze(2)
        batch_arange = torch.arange(B, device=x.device)
        h_history, y_list = [], []
        for t in range(L):
            valid_t = mask[:, t]
            p = parent_idx[:, t]
            has_parent = p >= 0
            if t == 0:
                h_prev = x.new_zeros(B, D, N)
            else:
                h_stack = torch.stack(h_history, dim=1)
                gathered = h_stack[batch_arange, p.clamp(min=0, max=t - 1)]
                h_prev = torch.where(has_parent.view(B, 1, 1), gathered, x.new_zeros(B, D, N))
            h_t = Abar[:, t] * h_prev + Bbar[:, t] * x[:, t].unsqueeze(-1)
            h_t = torch.where(valid_t.view(B, 1, 1), h_t, torch.zeros_like(h_t))
            h_history.append(h_t)
            y_t = (Ct[:, t].unsqueeze(1) * h_t).sum(-1) + self.D * x[:, t]
            y_t = torch.where(valid_t.view(B, 1), y_t, torch.zeros_like(y_t))
            y_list.append(y_t)
        return self.norm(torch.stack(y_list, dim=1) + x)


class HODMambaLite(nn.Module):
    def __init__(self, emb_dim, node_feat_dim, d_model=128, state_dim=16, num_layers=4,
                 n_sentiment=3, n_emotion=5):
        super().__init__()
        self.input_proj = nn.Linear(emb_dim, d_model)
        self.node_feat_proj = nn.Linear(node_feat_dim, d_model)
        self.layers = nn.ModuleList([TreeRecurrentSSMLayer(d_model, state_dim) for _ in range(num_layers)])
        self.root_context = nn.Parameter(torch.zeros(d_model))
        self.sentiment_head = nn.Linear(d_model, n_sentiment)
        self.emotion_head = nn.Linear(d_model, n_emotion)

    def forward(self, batch):
        x = self.input_proj(batch["embeddings"]) + self.node_feat_proj(batch["node_features"])
        parent_idx, mask = batch["parent_idx"], batch["mask"]
        for layer in self.layers:
            x = layer(x, parent_idx, mask)
        B, L, D = x.shape
        has_parent = parent_idx >= 0
        p = parent_idx.clamp(min=0)
        context = torch.stack([x[b, p[b]] for b in range(B)], dim=0)
        context = torch.where(has_parent.unsqueeze(-1), context, self.root_context.view(1, 1, D))
        return self.sentiment_head(context), self.emotion_head(context)


## Loss (mask passed explicitly) and evaluation, both mode-aware

In [6]:
def compute_loss(sent_logits, emo_logits, forecastable, batch):
    sent_valid = forecastable & (batch["sentiment_idx"] >= 0)
    emo_valid = forecastable & (batch["emotion_idx"] >= 0)
    loss = 0.0
    if sent_valid.any():
        loss = loss + F.cross_entropy(sent_logits[sent_valid], batch["sentiment_idx"][sent_valid])
    if emo_valid.any():
        loss = loss + F.cross_entropy(emo_logits[emo_valid], batch["emotion_idx"][emo_valid])
    return loss, sent_valid, emo_valid


def prepare_model_batch(batch, mode):
    '''Returns (model_input_batch, forecastable_mask). The forecastable mask always uses the
    TRUE tree parent, regardless of mode, so both conditions are scored on the same positions.'''
    tree_parent_idx = batch["parent_idx"]
    forecastable = batch["mask"] & (tree_parent_idx >= 0)
    if mode == "chain":
        model_batch = dict(batch)
        model_batch["parent_idx"] = make_chain_idx(batch["mask"])
    else:
        model_batch = batch
    return model_batch, forecastable


@torch.no_grad()
def evaluate(model, items, mode, batch_size=32):
    model.eval()
    rng = np.random.default_rng(0)
    batches = make_length_bucketed_batches(items, batch_size, rng)
    all_sp, all_st, all_ep, all_et = [], [], [], []
    for batch_idx in batches:
        batch = collate_fn([items[i] for i in batch_idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        model_batch, forecastable = prepare_model_batch(batch, mode)
        sl, el = model(model_batch)
        _, sv, ev = compute_loss(sl, el, forecastable, batch)
        all_sp.append(sl[sv].argmax(-1).cpu().numpy()); all_st.append(batch["sentiment_idx"][sv].cpu().numpy())
        all_ep.append(el[ev].argmax(-1).cpu().numpy()); all_et.append(batch["emotion_idx"][ev].cpu().numpy())
    sp, st = np.concatenate(all_sp), np.concatenate(all_st)
    ep, et = np.concatenate(all_ep), np.concatenate(all_et)
    return (f1_score(st, sp, average="macro", zero_division=0),
            f1_score(et, ep, average="macro", zero_division=0))


## Training loop 

In [7]:
def run_variant(mode, seed, batch_size=32, max_epochs=30, patience=5):
    torch.manual_seed(seed)
    model = HODMambaLite(
        emb_dim=embeddings.shape[1], node_feat_dim=node_feat_matrix.shape[1],
        n_sentiment=N_SENTIMENT, n_emotion=N_EMOTION,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    rng = np.random.default_rng(seed)

    best_combined_f1, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        batches = make_length_bucketed_batches(splits["train"], batch_size, rng)
        for batch_idx in batches:
            batch = collate_fn([splits["train"][i] for i in batch_idx])
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            model_batch, forecastable = prepare_model_batch(batch, mode)
            optimizer.zero_grad()
            sl, el = model(model_batch)
            loss, _, _ = compute_loss(sl, el, forecastable, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        val_sent_f1, val_emo_f1 = evaluate(model, splits["val"], mode, batch_size)
        combined_f1 = (val_sent_f1 + val_emo_f1) / 2
        if combined_f1 > best_combined_f1:
            best_combined_f1, patience_ctr = combined_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    model.load_state_dict(best_state)
    test_sent_f1, test_emo_f1 = evaluate(model, splits["test"], mode, batch_size)
    return test_sent_f1, test_emo_f1, epoch + 1


## Run 5 seeds x 2 conditions

In [8]:
SEEDS = [0, 1, 2, 3, 4]
results = {"tree": {"sentiment": [], "emotion": []}, "chain": {"sentiment": [], "emotion": []}}

for seed in SEEDS:
    for mode in ["tree", "chain"]:
        t0 = time.time()
        test_sent_f1, test_emo_f1, n_epochs = run_variant(mode, seed)
        results[mode]["sentiment"].append(test_sent_f1)
        results[mode]["emotion"].append(test_emo_f1)
        print(f"seed={seed} mode={mode:5s}: sentiment_f1={test_sent_f1:.3f}, emotion_f1={test_emo_f1:.3f}, "
              f"{n_epochs} epochs, {time.time()-t0:.0f}s")


seed=0 mode=tree : sentiment_f1=0.474, emotion_f1=0.319, 16 epochs, 561s
seed=0 mode=chain: sentiment_f1=0.462, emotion_f1=0.316, 16 epochs, 552s
seed=1 mode=tree : sentiment_f1=0.473, emotion_f1=0.328, 14 epochs, 489s
seed=1 mode=chain: sentiment_f1=0.443, emotion_f1=0.312, 14 epochs, 494s
seed=2 mode=tree : sentiment_f1=0.482, emotion_f1=0.319, 10 epochs, 351s
seed=2 mode=chain: sentiment_f1=0.452, emotion_f1=0.317, 15 epochs, 520s
seed=3 mode=tree : sentiment_f1=0.472, emotion_f1=0.325, 11 epochs, 383s
seed=3 mode=chain: sentiment_f1=0.466, emotion_f1=0.304, 10 epochs, 354s
seed=4 mode=tree : sentiment_f1=0.478, emotion_f1=0.319, 9 epochs, 316s
seed=4 mode=chain: sentiment_f1=0.463, emotion_f1=0.321, 13 epochs, 454s


## Aggregate: mean +/- std and a paired t-test

In [9]:
summary = {}
for task in ["sentiment", "emotion"]:
    tree_scores = np.array(results["tree"][task])
    chain_scores = np.array(results["chain"][task])
    t_stat, p_value = stats.ttest_rel(tree_scores, chain_scores)
    summary[task] = {
        "tree_mean": float(tree_scores.mean()), "tree_std": float(tree_scores.std()),
        "chain_mean": float(chain_scores.mean()), "chain_std": float(chain_scores.std()),
        "tree_scores": tree_scores.tolist(), "chain_scores": chain_scores.tolist(),
        "paired_t_stat": float(t_stat), "paired_p_value": float(p_value),
    }
    print(f"[{task}] tree:  {tree_scores.mean():.3f} +/- {tree_scores.std():.3f}  (seeds: {np.round(tree_scores,3).tolist()})")
    print(f"[{task}] chain: {chain_scores.mean():.3f} +/- {chain_scores.std():.3f}  (seeds: {np.round(chain_scores,3).tolist()})")
    print(f"[{task}] paired t-test: t={t_stat:.3f}, p={p_value:.4f}"
          f"  -> {'significant at alpha=0.05' if p_value < 0.05 else 'NOT significant at alpha=0.05'}\n")


[sentiment] tree:  0.476 +/- 0.004  (seeds: [0.474, 0.473, 0.482, 0.472, 0.478])
[sentiment] chain: 0.457 +/- 0.008  (seeds: [0.462, 0.443, 0.452, 0.466, 0.463])
[sentiment] paired t-test: t=3.916, p=0.0173  -> significant at alpha=0.05

[emotion] tree:  0.322 +/- 0.004  (seeds: [0.319, 0.328, 0.319, 0.325, 0.319])
[emotion] chain: 0.314 +/- 0.006  (seeds: [0.316, 0.312, 0.317, 0.304, 0.321])
[emotion] paired t-test: t=1.810, p=0.1446  -> NOT significant at alpha=0.05



In [10]:
MODELS_DIR = os.path.join(BASE_DIR, "processed_v2", "models")
os.makedirs(MODELS_DIR, exist_ok=True)
with open(os.path.join(MODELS_DIR, "ablation_chain_vs_tree.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("Saved:", os.path.join(MODELS_DIR, "ablation_chain_vs_tree.json"))


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/models/ablation_chain_vs_tree.json
